# Experimentos — Transcrição de tablatura robusta a efeitos

Notebook único e **portátil**: roda em **Colab** ou **local com GPU** trocando só
`PLATFORM` na 1ª célula. Rode de cima para baixo.

- **Colab**: a cada sessão nova, rode as células de *setup* (reseta tudo em `/content`).
- **Local (1ª vez)**: rode dentro de um venv com PyTorch+CUDA; o setup clona os
  frameworks em `../frameworks` e instala no venv **uma vez**. Depois, só verifica.

Ordem: **config → setup → dados → smoke → rodada completa → comparação (0d)**.

## 1. Configuração (edite aqui)

In [ ]:
PLATFORM     = "AUTO"        # "AUTO" | "COLAB" | "LOCAL"
GITHUB_USER  = "abUFS"
PROJECT_NAME = "timbre-robust-gtt"          # nome do seu repositório
CONDITION    = "configs/baseline_tabcnn.yaml"
WITH_FRETNET = False        # True só na Fase 4 (FretNet)

import os, sys, subprocess

# resolve a plataforma
_colab = ("google.colab" in sys.modules) or os.path.isdir("/content")
plat = PLATFORM.upper()
plat = ("COLAB" if _colab else "LOCAL") if plat == "AUTO" else plat
os.environ["TCC_PLATFORM"] = plat

# garante o código do projeto disponível e o cwd na raiz do repo
if plat == "COLAB":
    repo = f"/content/{PROJECT_NAME}"
    if not os.path.isdir(repo):
        subprocess.run(["git", "clone", "--depth", "1",
                        f"https://github.com/{GITHUB_USER}/{PROJECT_NAME}.git",
                        repo], check=True)
    os.chdir(repo)
else:
    here = os.getcwd()
    repo = here if os.path.isdir(os.path.join(here, "src")) else os.path.dirname(here)
    os.chdir(repo)
sys.path.insert(0, os.getcwd())
print("plataforma:", plat, "| repo:", os.getcwd())

## 2. Setup do ambiente

Uma chamada resolve tudo: Colab monta Drive + clona + instala; local só verifica
(ou instala na 1ª vez). Ao final, imprime imports + GPU + caminhos.

In [ ]:
from src.env_setup import setup
setup(platform=os.environ["TCC_PLATFORM"], with_fretnet=WITH_FRETNET)

## 3. Dados — GuitarSet

Baixa `annotation.zip` + `audio_mono-mic.zip` do Zenodo para o "armazém"
(verifica MD5) e extrai para a "bancada" local. Idempotente: rodadas seguintes
pulam o download e só remontam a bancada em segundos.

In [ ]:
from src import download_guitarset as dg
dg.main()

## 4. Smoke run (validar o pipeline)

~60 iterações, 1 dobra. Não é instantâneo (gera o cache CQT), mas confirma que
tudo roda ponta-a-ponta e deixa o cache pronto.

In [ ]:
from src.train import run_condition
run_condition(CONDITION, quick=True)

## 5. Rodada completa (2500 iterações × 6 dobras)

Leva horas. **É retomável**: se a sessão resetar (Colab), reexecute esta mesma
célula — dobras concluídas são puladas, e a dobra em andamento retoma do último
checkpoint (tudo no "armazém"). O cache CQT é restaurado do armazém, sem recomputar.

In [ ]:
run_condition(CONDITION)

## 6. Comparação 0d — publicado vs. reproduzido

Compara o `summary.csv` das 6 dobras com os números publicados do TabCNN e dá o
veredito da Fase 0.

In [ ]:
from src.compare_baseline import compare
compare()

---
### Notas

- **Reset do Colab**: só o "armazém" (Drive) sobrevive. Reexecute as células 1-3
  a cada sessão nova antes de treinar.
- **Local**: nada reseta; após a 1ª execução, as células 1-3 são quase instantâneas.
- **FretNet (Fase 4)**: ligue `WITH_FRETNET=True`. Se o `muda` brigar no Python
  3.12, use `condacolab` (Python 3.10) — ver README.